# Master equations: experiments

Select the **me** conda environment as the kernel. Lengths are nm, times are μs, and densities are nm⁻³. This notebook uses the installed `master_equations` package and writes exports under `results/notebook/` in the current directory.

In [ ]:
import sys
from pathlib import Path

assert Path(sys.prefix).name == "me", "Select the me conda environment"
print(f"Interpreter: {sys.executable}")

In [ ]:
%matplotlib inline

from dataclasses import replace

from master_equations import (
    MasterEquation,
    Numerics,
    Parameters,
    Result,
    estimate_annihilation,
    solve_steady_state,
    solve_transient,
    sweep,
)
from master_equations.plotting import plot_correlations, plot_overview, save_figure

## Transient decay

The final radial bin is held at g=1 as a far-field reservoir. Refine the grid, domain, and quadrature independently before using numerical results in research.

In [ ]:
parameters = Parameters(tta_radius=3.5, tpq_radius=3.5, dexter_rate=30.0)
numerics = Numerics(bins=24, r_max=60, quadrature_order=32)
model = MasterEquation(parameters, numerics, cache_dir=".cache/master-equations")
transient = solve_transient(model, t_end=10, samples=201)
transient.summary()

In [ ]:
plot_overview(transient)

In [ ]:
plot_correlations(transient)

## Luminescence estimates

These reproduce the old apparent `kTT2` (yield) and `kTT1` (half-yield time) conventions. Their rate law includes a factor of 1/2. With TPQ present they combine quenching mechanisms; they do not identify TTA independently.

In [ ]:
estimate = estimate_annihilation(transient)
{"kTT2": estimate.from_yield, "kTT1": estimate.from_half_time, "ratio": estimate.ratio}

## Driven steady state

Convergence checks every density and correlation derivative. Integrated losses continue increasing at equilibrium and are excluded from the residual.

In [ ]:
driven = MasterEquation(
    replace(parameters, generation_rate=0.01),
    numerics,
    cache_dir=".cache/master-equations",
)
steady = solve_steady_state(driven, max_time=1000)
steady.summary()

In [ ]:
plot_overview(steady)

## Parameter sweep

Each parameter object is independent, and all three experiments reuse the same geometry cache.

In [ ]:
points = list(
    sweep(
        parameters,
        {"dexter_rate": [0.0, 10.0, 100.0]},
        numerics=numerics,
        cache_dir=".cache/master-equations",
        t_end=10,
        samples=101,
    )
)
[{**point.values, **point.result.summary()} for point in points]

## Save and reload

NPZ preserves all state arrays and parameters without pickle. Excel includes correlations and metadata; SVG can be edited or included in a thesis.

In [ ]:
output = Path("results/notebook")
transient.save(output / "transient.npz")
transient.to_csv(output / "transient.csv")
transient.to_excel(output / "transient.xlsx")
save_figure(plot_overview(transient), output / "transient.svg")
restored = Result.load(output / "transient.npz")
assert restored.parameters == parameters
restored.summary()